[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charles9257/PhD-Datasets-Analysis/blob/claude/project-thread-9pn62q/notebooks/EmoDB/EmoDB_Explore.ipynb)

# EmoDB: load, explore and probe

Berlin Database of Emotional Speech: 535 German utterances from 10 actors (5 F / 5 M, aged 21 to 35), 7 emotions.

This notebook works on this one dataset only. It builds a metadata table (path, emotion, speaker, gender, age), shows distributions, durations and audio examples, extracts MFCC / log-mel / wav2vec2 features, and runs quick probes: an emotion baseline (UAR, macro-F1) and attribute probes (gender, speaker ID, age where available) as a first leakage check.

**Dataset notes**
* Small corpus: results vary a lot between splits.
* Age range is narrow (21 to 35), so age probes here say little.

**Source data is never modified.** Outputs go to `OUT_DIR` (on Colab: `MyDrive/SER_outputs/EmoDB`).

**On Google Colab:** Runtime > Change runtime type > **T4 GPU**, then run cells top to bottom.

## 0. Environment setup (Colab or local)

Mounts Google Drive on Colab, installs missing packages and checks for a GPU.

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Colab already has most packages; this makes sure the audio ones are present and recent enough.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "librosa>=0.10", "soundfile", "transformers>=4.40", "ipywidgets"], check=True)
else:
    print("Not on Colab: skipping Drive mount. Install dependencies once with: pip install -r requirements.txt")

try:
    import torch
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    else:
        print("No GPU found, running on CPU. On Colab: Runtime > Change runtime type > T4 GPU, then re-run from the top.")
except ImportError:
    print("PyTorch not installed: the SSL embedding step will be skipped.")

## 1. Configuration

`DATA_ROOT` should mirror your local `Datasets` folder (default on Colab: `MyDrive/Datasets`).

In [ ]:
from pathlib import Path
import os, re, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATASET = "EmoDB"

# ---- Paths ----
# DATA_ROOT mirrors the local folder C:\Desktop\RGU PhD\New folder\Datasets.
# On Colab, copy/sync that folder to Google Drive as MyDrive/Datasets, or change the path below.
IN_COLAB = "google.colab" in __import__("sys").modules
DEFAULT_ROOT = "/content/drive/MyDrive/Datasets" if IN_COLAB else r"C:\Desktop\RGU PhD\New folder\Datasets"
DATA_ROOT = Path(os.environ.get("DATA_ROOT", DEFAULT_ROOT))

PATHS = {
    "EMODB_WAV":  DATA_ROOT / "Datasets" / "EmoDB",
    "EMODB_DEMO": DATA_ROOT / "Datasets" / "emoDB demographics - Sheet1.csv",
}
REQUIRED = ["EMODB_WAV", "EMODB_DEMO"]

# ---- Outputs (never inside the source folders) ----
if IN_COLAB:
    OUT_DIR = Path("/content/drive/MyDrive/SER_outputs") / DATASET
else:
    try:
        NB_DIR = Path(__vsc_ipynb_file__).parent          # VS Code
    except NameError:
        NB_DIR = Path.cwd()                               # Jupyter: folder the notebook was opened from
    OUT_DIR = NB_DIR / "outputs"
(OUT_DIR / "features").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

# ---- Knobs ----
SEED = 42
N_SAMPLE = 600              # utterances sampled (spread across speakers) for features/probes; None = all (slow)
MAX_SECONDS = 6.0           # crop long utterances
TARGET_SR = 16_000
RUN_SSL = True              # wav2vec2 embeddings (GPU recommended); set False for handcrafted features only
SSL_MODEL = "facebook/wav2vec2-base"   # or "facebook/hubert-base-ls960"

random.seed(SEED); np.random.seed(SEED)
for k, p in PATHS.items():
    print(f"{k:16s} {'OK ' if p.exists() else 'MISSING'}  {p}")
print("Outputs ->", OUT_DIR)
if not all(PATHS[k].exists() for k in REQUIRED):
    raise FileNotFoundError(f"{DATASET} files not found under {DATA_ROOT}. Check the Drive folder or edit DATA_ROOT / PATHS above.")

## 2. Metadata table

One row per utterance: `path, corpus, emotion, emotion_raw, speaker, gender, age, extra`.

In [ ]:
def load_dataset():
    demo = pd.read_csv(PATHS["EMODB_DEMO"]).set_index("ActorID")
    m = {"W":"angry","L":"boredom","E":"disgust","A":"fear","F":"happy","T":"sad","N":"neutral"}
    rows = []
    for f in sorted(PATHS["EMODB_WAV"].glob("*.wav")):
        spk, emo = int(f.stem[:2]), f.stem[5]
        d = demo.loc[spk]
        rows.append(dict(path=str(f), corpus="EmoDB", emotion=m[emo], emotion_raw=emo, speaker=f"EMO_{spk:02d}",
                         gender=d.Gender.lower(), age=float(d.Age), extra=f"text={f.stem[2:5]}"))
    return pd.DataFrame(rows)

In [ ]:
meta = load_dataset()
meta.to_csv(OUT_DIR / f"{DATASET}_metadata.csv", index=False)
HAS_EMO = meta.emotion.notna().any()
HAS_GENDER = meta.gender.dropna().nunique() > 1
HAS_AGE = meta.age.notna().any() and meta.drop_duplicates("speaker").age.nunique() > 1

print(f"{DATASET}: {len(meta):,} utterances, {meta.speaker.nunique()} speakers")
print("Gender (speakers):", meta.drop_duplicates("speaker").gender.value_counts(dropna=False).to_dict())
if HAS_EMO: print("Emotions:", meta.emotion.value_counts().to_dict())
speakers = meta.groupby("speaker").agg(gender=("gender", "first"), age=("age", "first"), utterances=("path", "size"))
display(meta.head())
speakers

## 3. Exploration

In [ ]:
panels = [p for p, ok in [("emotion", HAS_EMO), ("speaker", True), ("emo_gender", HAS_EMO and HAS_GENDER), ("age", HAS_AGE)] if ok]
fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4.5), squeeze=False); axes = axes[0]
for ax, p in zip(axes, panels):
    if p == "emotion":
        meta.emotion.value_counts().plot.bar(ax=ax, color="tab:blue"); ax.set_title("Utterances per emotion")
    elif p == "speaker":
        top = speakers.sort_values("utterances", ascending=False).head(40)
        top.utterances.plot.bar(ax=ax, color=top.gender.map({"female": "tab:red", "male": "tab:blue"}).fillna("grey").tolist())
        ax.set_title("Utterances per speaker (red F, blue M, grey unknown)"); ax.set_xticklabels([])
    elif p == "emo_gender":
        pd.crosstab(meta.emotion, meta.gender).plot.bar(ax=ax); ax.set_title("Emotion x gender")
    elif p == "age":
        speakers.age.dropna().plot.hist(bins=15, ax=ax); ax.set_title("Speaker age")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / "distributions.png", dpi=150); plt.show()

if HAS_EMO and HAS_GENDER:
    # Emotion and gender are often correlated in acted corpora; this matters for attribute leakage
    display(pd.crosstab(meta.emotion, meta.gender, normalize="index").round(2))

In [ ]:
import librosa, librosa.display, soundfile as sf
from IPython.display import Audio, display

def info(p):
    try:
        i = sf.info(p); return i.duration, i.samplerate
    except Exception:
        return librosa.get_duration(path=p), librosa.get_samplerate(p)

samp = meta.sample(min(len(meta), 300), random_state=SEED).copy()
samp[["dur", "sr"]] = [info(p) for p in samp.path]
print("Sample rates:", samp.sr.value_counts().to_dict())
samp.groupby("emotion" if HAS_EMO else "gender", dropna=False).dur.describe().round(2)

In [ ]:
def load_audio(path, sr=TARGET_SR, max_s=None):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_s)
    return y

def show_example(row):
    y = load_audio(row.path, max_s=10)
    fig, ax = plt.subplots(1, 2, figsize=(14, 3))
    librosa.display.waveshow(y, sr=TARGET_SR, ax=ax[0]); ax[0].set_title(f"{row.emotion} | {row.speaker} | {row.gender}")
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=64), ref=np.max)
    librosa.display.specshow(S, sr=TARGET_SR, x_axis="time", y_axis="mel", ax=ax[1]); ax[1].set_title("log-mel")
    plt.show(); display(Audio(y, rate=TARGET_SR))

by = "emotion" if HAS_EMO else "speaker"
for _, row in meta.dropna(subset=[by]).groupby(by).sample(1, random_state=SEED).head(8).iterrows():
    show_example(row)

## 4. Preprocessing and features

16 kHz mono, silence trim (`top_db=30`), peak normalisation, crop to `MAX_SECONDS`. MFCC (40 + deltas, mean/std pooled, 160-d), log-mel (64 bands, mean/std, 128-d), and optionally wav2vec2 last-layer mean-pooled embeddings (all layers are cached). Cached in `OUT_DIR/features`.

In [ ]:
from tqdm.auto import tqdm

def sample_meta(meta, n=N_SAMPLE):
    # sample utterances spreading across speakers (and emotions where available)
    if n is None or n >= len(meta): return meta.reset_index(drop=True)
    per = max(1, n // meta.speaker.nunique())
    s = pd.concat([g.sample(min(len(g), per), random_state=SEED) for _, g in meta.groupby("speaker")])
    if len(s) < n:
        s = pd.concat([s, meta.drop(s.index).sample(min(n - len(s), len(meta) - len(s)), random_state=SEED)])
    return s.sample(min(n, len(s)), random_state=SEED).reset_index(drop=True)

work = sample_meta(meta)
print(len(work), "utterances,", work.speaker.nunique(), "speakers")

def preprocess(path):
    y = load_audio(path)
    y, _ = librosa.effects.trim(y, top_db=30)
    y = y / (np.abs(y).max() + 1e-8)
    return y[: int(MAX_SECONDS * TARGET_SR)]

def handcrafted(y):
    mf = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=40)
    mf = np.vstack([mf, librosa.feature.delta(mf)])
    lm = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_mels=64))
    return np.concatenate([mf.mean(1), mf.std(1)]), np.concatenate([lm.mean(1), lm.std(1)])

tag = f"n{N_SAMPLE}_s{SEED}"
f_cache = OUT_DIR / "features" / f"handcrafted_{tag}.npz"
if f_cache.exists():
    z = np.load(f_cache); X_mfcc, X_mel = z["mfcc"], z["mel"]
else:
    mf, ml = [], []
    for p in tqdm(work.path, desc="MFCC/log-mel"):
        a, b = handcrafted(preprocess(p)); mf.append(a); ml.append(b)
    X_mfcc, X_mel = np.array(mf), np.array(ml)
    np.savez(f_cache, mfcc=X_mfcc, mel=X_mel)
work.to_csv(OUT_DIR / "features" / f"work_meta_{tag}.csv", index=False)
FEATS = {"MFCC": X_mfcc, "log-mel": X_mel}
{k: v.shape for k, v in FEATS.items()}

In [ ]:
if RUN_SSL:
    import torch
    from transformers import AutoFeatureExtractor, AutoModel
    device = "cuda" if torch.cuda.is_available() else "cpu"
    short = SSL_MODEL.split("/")[-1]
    cache = OUT_DIR / "features" / f"{short}_{tag}.npy"
    if cache.exists():
        E = np.load(cache)
    else:
        fe = AutoFeatureExtractor.from_pretrained(SSL_MODEL)
        model = AutoModel.from_pretrained(SSL_MODEL, output_hidden_states=True).to(device).eval()
        out = []
        with torch.no_grad():
            for p in tqdm(work.path, desc=short):
                inp = fe(preprocess(p), sampling_rate=TARGET_SR, return_tensors="pt").to(device)
                hs = model(**inp).hidden_states                                   # tuple(L+1) of [1, T, 768]
                out.append(torch.stack([h.mean(1)[0] for h in hs]).cpu().numpy())  # [L+1, 768]
        E = np.stack(out); np.save(cache, E)                                     # [N, L+1, 768]
    FEATS[short] = E[:, -1]
    print(short, E.shape)
else:
    print("RUN_SSL = False: using handcrafted features only")

## 5. Quick probes

Emotion recognition baseline and attribute probes (what an attacker could recover from the representations). Probes that the data cannot support (for example gender on a single-gender corpus) are skipped with a message.

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score, mean_absolute_error

def clf(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))

def split(y, groups, speaker_independent, test_size=0.25):
    """Speaker-independent split with >= 2 classes on both sides (tries several seeds); None if impossible."""
    for s in range(SEED, SEED + 30):
        if speaker_independent:
            tr, te = next(GroupShuffleSplit(1, test_size=test_size, random_state=s).split(y, y, groups))
        else:
            tr, te = next(StratifiedShuffleSplit(1, test_size=test_size, random_state=s).split(y, y))
        if len(set(y[tr])) > 1 and len(set(y[te])) > 1 and set(y[te]) <= set(y[tr]):
            return tr, te
    return None

def eval_task(X, y, groups, speaker_independent=True):
    y = np.asarray(y); sp = split(y, np.asarray(groups), speaker_independent)
    if sp is None: return None
    tr, te = sp; m = clf().fit(X[tr], y[tr]); p = m.predict(X[te])
    maj = pd.Series(y[tr]).mode()[0]
    acc, base = accuracy_score(y[te], p), accuracy_score(y[te], np.full(len(te), maj))
    return dict(UAR=balanced_accuracy_score(y[te], p), F1=f1_score(y[te], p, average="macro"),
                acc=acc, majority_acc=base, advantage=acc - base)

res = []
for fname, X in FEATS.items():
    tasks = []
    if HAS_EMO:
        ok = work.emotion.notna().values
        tasks.append(("emotion (unseen speakers)", ok, work.emotion.values, True))
    if HAS_GENDER:
        ok = work.gender.notna().values
        tasks.append(("gender (unseen speakers)", ok, work.gender.values, True))
    counts = work.speaker.map(work.speaker.value_counts())
    if work.speaker.nunique() > 1:
        ok = (counts >= 4).values
        tasks.append(("speaker ID (closed set)", ok, work.speaker.values, False))
    for task, ok, y, si in tasks:
        r = eval_task(X[ok], y[ok], work.speaker.values[ok], si)
        if r is None:
            print(f"skip {task}: not enough speakers/classes for a valid split"); continue
        r.update(task=task, features=fname); res.append(r)
probe = pd.DataFrame(res)
if len(probe):
    probe = probe[["task", "features", "UAR", "F1", "acc", "majority_acc", "advantage"]].round(3)
    probe.to_csv(OUT_DIR / f"probes_{tag}.csv", index=False)
probe

In [ ]:
# Age probe: regression MAE on unseen speakers (only if the dataset has age labels for several speakers)
ok = work.age.notna().values
if HAS_AGE and work[ok].speaker.nunique() >= 4:
    g = work[ok]; yv = g.age.values
    tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=SEED).split(g, groups=g.speaker))
    for fname, X in FEATS.items():
        Xi = X[ok]
        m = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xi[tr], yv[tr])
        print(f"{fname:16s} age MAE = {mean_absolute_error(yv[te], m.predict(Xi[te])):.1f} y   "
              f"(predict-mean MAE = {mean_absolute_error(yv[te], np.full(len(te), yv[tr].mean())):.1f})")
else:
    print("Age probe skipped: no age labels or fewer than 4 speakers with age")

## Notes

* Metadata for this corpus is saved as `{DATASET}_metadata.csv` in `OUT_DIR`, with the same columns as the combined notebook (`path, corpus, emotion, emotion_raw, speaker, gender, age, extra`), so tables from the per-dataset notebooks can be concatenated.
* **Speaker-independent** splits are used for emotion and gender; speaker ID is closed-set by definition.
* Attacker **advantage** = accuracy minus majority-class accuracy. Proposed gate: a defence passes if advantage stays within 5 pp while emotion UAR drops by at most 5 points (pending supervisor confirmation).
* For cross-corpus experiments and the FedAvg attack simulation, use the combined notebook `notebooks/SER_Privacy_Dataset_Prep.ipynb`.